## 1. Imports

In [ ]:
import os
import time
import json
import requests
from pathlib import Path
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.agents import AgentsClient              # ✅
from azure.ai.agents.models import FunctionTool

In [ ]:
# -----------------------------
# Load credentials
# -----------------------------
def find_cred_json(start_path):
    current = Path(start_path)
    while current != current.parent:
        cred_file = current / 'cred.json'
        if cred_file.exists():
            return str(cred_file)
        current = current.parent
    return None
 
try:
    file_path = find_cred_json(os.getcwd())
    if not file_path:
        raise FileNotFoundError("Could not find cred.json file")
 
    print(f"Found cred.json at: {file_path}")
 
    with open(file_path, 'r') as f:
        loaded_config = json.load(f)
 
    project_client = AIProjectClient(
        credential=DefaultAzureCredential(),
        endpoint=loaded_config["PROJECT_ENDPOINT"],
    )
 
    agents_client = AgentsClient(                     # ✅
        credential=DefaultAzureCredential(),
        endpoint=loaded_config["PROJECT_ENDPOINT"],
    )
 
    model_name = loaded_config.get("MODEL_DEPLOYMENT_NAME", "gpt-4o")
    print("✅ Successfully initialized AIProjectClient and AgentsClient")
 
except Exception as e:
    print(f"❌ Error initializing clients: {e}")
    exit(1)

## 2. Define helper functions

In [ ]:
# -----------------------------
# Function 1: Weather (OpenWeatherMap)
# -----------------------------
def fetch_weather(location: str) -> str:
    api_key = "your-openweathermap-api-key"
    base_url = "http://api.openweathermap.org/data/2.5/weather"
    try:
        params = {'q': location, 'appid': api_key, 'units': 'metric'}
        response = requests.get(base_url, params=params)
        response.raise_for_status()
        data = response.json()
 
        temperature     = data['main']['temp']
        weather_condition = data['weather'][0]['description'].capitalize()
        humidity        = data['main']['humidity']
        wind_speed      = data['wind']['speed'] * 3.6  # m/s → km/h
 
        weather_info = f"{weather_condition}, {temperature}°C, Humidity: {humidity}%, Wind: {wind_speed:.1f} km/h"
 
        return json.dumps({
            "weather": weather_info,
            "location": location,
            "temperature": temperature,
            "condition": weather_condition,
            "humidity": humidity,
            "wind_speed": round(wind_speed, 1)
        })
    except Exception as e:
        return json.dumps({"error": f"Error fetching weather data: {str(e)}", "location": location})

In [ ]:
# -----------------------------
# Function 2: Exchange Rate
# -----------------------------
def get_exchange_rate(from_currency: str, to_currency: str, amount: float = 1.0) -> str:
    try:
        url = f"https://api.exchangerate-api.com/v4/latest/{from_currency}"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()
        rate = data['rates'].get(to_currency)
        if not rate:
            return json.dumps({"error": f"Currency {to_currency} not found"})
        converted = amount * rate
        return json.dumps({
            "from_currency": from_currency,
            "to_currency": to_currency,
            "amount": amount,
            "rate": rate,
            "converted_amount": round(converted, 2)
        })
    except Exception as e:
        return json.dumps({"error": f"Error fetching exchange rate: {str(e)}"})

In [ ]:
# -----------------------------
# Function 3: Math Calculator
# -----------------------------
def calculate_math(expression: str) -> str:
    try:
        result = eval(expression, {"__builtins__": {}})
        return json.dumps({"expression": expression, "result": result})
    except Exception as e:
        return json.dumps({"error": f"Error calculating expression: {str(e)}"})

In [ ]:
# -----------------------------
# Setup FunctionTool
# -----------------------------
user_functions = {fetch_weather, get_exchange_rate, calculate_math}
functions = FunctionTool(functions=user_functions)
 

In [ ]:
# -----------------------------
# Create agent
# -----------------------------
def create_multi_function_agent():
    try:
        agent = agents_client.create_agent(           # ✅
            model=model_name,
            name="multi-function-agent",
            instructions="""You are a helpful assistant with multiple capabilities:
            1. Weather: Use fetch_weather to get weather information for any location
            2. Currency: Use get_exchange_rate to convert between currencies
            3. Math: Use calculate_math to evaluate mathematical expressions
            
            Always use the appropriate function based on the user's request.""",
            tools=functions.definitions,
        )
        print(f"✅ Created agent, ID: {agent.id}")
        return agent
    except Exception as e:
        print(f"❌ Error creating agent: {e}")
        return None

In [ ]:
# -----------------------------
# Run conversation
# -----------------------------
def run_conversation(agent, user_message):
    try:
        thread = agents_client.threads.create()       # ✅
        print(f"📝 Created thread, ID: {thread.id}")
 
        message = agents_client.messages.create(      # ✅
            thread_id=thread.id,
            role="user",
            content=user_message,
        )
        print(f"➕ Created message, ID: {message.id}")
 
        run = agents_client.runs.create(              # ✅
            thread_id=thread.id,
            agent_id=agent.id
        )
        print(f"▶️ Created run, ID: {run.id}")
 
        max_attempts = 60
        attempts = 0
 
        while run.status in ["queued", "in_progress", "requires_action"] and attempts < max_attempts:
            time.sleep(1)
            attempts += 1
            run = agents_client.runs.get(             # ✅
                thread_id=thread.id,
                run_id=run.id
            )
 
            if run.status == "requires_action":
                print("🔧 Processing function calls...")
                tool_calls = run.required_action.submit_tool_outputs.tool_calls
                tool_outputs = []
 
                for tool_call in tool_calls:
                    function_name = tool_call.function.name
                    args = json.loads(tool_call.function.arguments)
 
                    print(f"   📞 Calling function: {function_name}")
                    print(f"      Arguments: {args}")
 
                    if function_name == "fetch_weather":
                        output = fetch_weather(args.get("location", "New York"))
                    elif function_name == "get_exchange_rate":
                        output = get_exchange_rate(
                            args.get("from_currency", "USD"),
                            args.get("to_currency", "EUR"),
                            args.get("amount", 1.0)
                        )
                    elif function_name == "calculate_math":
                        output = calculate_math(args.get("expression", "0"))
                    else:
                        output = json.dumps({"error": f"Unknown function: {function_name}"})
 
                    tool_outputs.append({
                        "tool_call_id": tool_call.id,
                        "output": output
                    })
 
                agents_client.runs.submit_tool_outputs( # ✅
                    thread_id=thread.id,
                    run_id=run.id,
                    tool_outputs=tool_outputs
                )
                print("   ✅ Submitted tool outputs")
 
        print(f"🤖 Run completed with status: {run.status}")
 
        if run.status == "completed":
            messages = agents_client.messages.list(thread_id=thread.id)  # ✅
            message_list = list(messages)
 
            print("\n💬 Conversation:")
            for message in message_list:
                role = message.role
                if message.content:
                    for content in message.content:
                        if hasattr(content, 'text') and content.text:
                            print(f"   {role.upper()}: {content.text.value}")
 
        return thread, run
 
    except Exception as e:
        print(f"❌ Error in conversation: {e}")
        import traceback
        traceback.print_exc()
        return None, None
 

In [ ]:
# -----------------------------
# Main execution
# -----------------------------
if __name__ == "__main__":
    agent = create_multi_function_agent()
 
    if agent:
        test_queries = [
            "What's the weather in Tokyo and Paris?",
            "Convert 100 USD to EUR and GBP",
            "Calculate (25 * 4) + (100 / 2) - 10"
        ]
 
        for i, query in enumerate(test_queries, 1):
            print(f"\n{'='*60}")
            print(f"🗣️ Test Query {i}: {query}")
            print(f"{'='*60}")
 
            thread, run = run_conversation(agent, query)
 
            if i < len(test_queries):
                print("\n⏳ Waiting 2 seconds before next query...")
                time.sleep(2)
 
        try:
            agents_client.delete_agent(agent.id)      # ✅
            print("\n🗑️ Deleted agent")
        except Exception as e:
            print(f"⚠️ Could not delete agent: {e}")
 